# Storage Change — Direct-to-Volume Streaming (GridFS Deprecated)

**System:** File Validation & Routing Framework • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW • **Supersedes:** HLD decision **D2** (single GridFS bucket as the bytes store) • **Companion updates:** LLD v1.3, Build Steps 1–2 (patched), Purge Job Design (banner note)

## 1. Decision & Rationale

**Portal uploads now stream directly to a Databricks UC staging volume; GridFS is removed from Phase 1.** MongoDB remains the metadata system of record; volumes hold the bytes.

Why GridFS is the wrong tool for this workload (files mostly < 15 MB, chunk = 255 KB):

| GridFS cost | Number for a 10 MB file | Impact |
| --- | --- | --- |
| `fs.chunks` documents | ~40 (255 KB chunks) | ~40 writes + index entries per file, vs **1 PUT** to a volume path |
| Purge delete | deleteMany 40 chunk docs in a transaction | heavy; volume purge = **1 DELETE** |
| Bytes never touched by validation | still round-trip GridFS → FastAPI worker | volume staging lets validation and intake live on the **same storage tier** |
| Value of Mongo-managed bytes | low — files already flow to Databricks volumes for intake | double copy, double lifecycle |

**What does NOT change:** MongoDB as the authoritative metadata + control plane (`file_registry` state machine, rules-as-data, events, audit, idempotency via unique indexes), the event-log-centric pipeline, the guarded-atomic write patterns, reconciliation, and the purge gate (retention exceeded AND MATCHED). Only the bytes' home changes.

## 2. Verified volume file mechanics (Files API, `/api/2.0/fs/*`)

All operations take an absolute path — `/Volumes/<catalog>/<schema>/<volume>/...` works directly; no bucket/chunk bookkeeping:

| Operation | Method & path | Semantics |
| --- | --- | --- |
| Upload | `PUT /api/2.0/fs/files{file_path}` (raw body) | `overwrite` param: **default true = idempotent re-PUT** (last write wins); `overwrite=false` errors if the path exists — use as a finalize guard |
| Download | `GET /api/2.0/fs/files{file_path}` | full-file stream (fine for < 15 MB validation reads) |
| Metadata | `HEAD /api/2.0/fs/files{file_path}` | existence + size — the staging finalize check |
| Delete | `DELETE /api/2.0/fs/files{file_path}` | staging cleanup after a successful move; purge |
| Directories | `PUT /api/2.0/fs/directories{path}` (create), `GET` (list), `DELETE` | staging per-session dirs; avoid `GET` list on large dirs — all access is path-addressable |

**Security:** the FastAPI service principal (OAuth, token from vault — never in the repo) needs `WRITE VOLUME` on the staging volume and `WRITE FILES`/`READ FILES` on the staging/intake/quarantine roots; least-privilege per volume. **Resumability:** single PUT per file — a dropped 10 MB upload is simply re-PUT (whole-file retry, acceptable under the 15 MB profile; revisit chunked sessions only if files grow past ~100 MB — threshold documented, not built).

## 3. Revised Flow — Upload → Validate → Route (all bytes stay in volumes)

```text
React ──HTTPS──► FastAPI (stateless pods)
                    │ 1. POST /manifest          → upload_sessions (IN_PROGRESS, declared files)
                    │ 2. stream each file (single pass):
                    │      • pre-staging checks: FILE_TYPE_ALLOWED, FILE_SIZE_MAX (Content-Length),
                    │        FILE_NOT_EMPTY (declared size) — reject BEFORE any bytes are written
                    │      • sha256 + size computed on the fly as bytes pass through
                    │      • PUT /api/2.0/fs/files /Volumes/…/staging/sessions/<ses_id>/<original_name>
                    │ 3. complete: HEAD staged file (size match) → file_registry insert
                    │      (staged_path + sha256, unique-index dedup) → FILE_ARRIVED event
                    │      → file_lifecycle {location: STAGING_VOLUME}
                    ▼
               MongoDB event_log (unchanged envelope — SharePoint/SFTP adapters slot in later)
                    ▼
               pipeline worker (event-driven, N workers)
                    │ 4. GET staged file (small-file download), run HEADER + CONTENT rules
                    │      → validation_results (unchanged evidence model)
                    │ 5. routing policy → file_transfers (PENDING, partial-unique COMPLETED guard)
                    │ 6. move: GET staging → PUT volume_map-resolved INTAKE/QUARANTINE path
                    │      (renamed {file_type}_{file_id}_{ts}.ext) → verify size → DELETE staging copy
                    │      → transfer COMPLETED → file_lifecycle {INTAKE_VOLUME|QUARANTINE}
                    │      → FILE_ROUTED + TRANSFER_COMPLETED events
```

**Staging layout:** `staging/sessions/<session_id>/<original_name>` — per-session directories make paths collision-free by construction and give the session janitor a single directory to clean for abandoned uploads (`DELETE /api/2.0/fs/directories`, replacing orphan-chunk cleanup).

**Validation timing stays rule-registry-driven** — same 8 seed rules, same `validation_rules`/`validation_results` model, same R2 dry-run. Only where each rule executes shifts: FILE-category rules pre-staging (cheap manifest/length checks, reject before writing bytes) + duplicate at finalize (sha256); HEADER/CONTENT post-staging. A rejected-in-flight file never touches storage — the 409/duplicate responses come back before the PUT.

## 4. Concurrency & Reliability Analysis (does it hold up?)

| Concern | Verdict | Why |
| --- | --- | --- |
| Many simultaneous uploads | ✅ | each file is an **independent object PUT** to object storage — no shared chunks, no write contention, no cross-file locks; scale = add FastAPI pods |
| Two uploads collide on one file | ✅ impossible | staging paths are `sessions/<session_id>/…` — unique by construction; re-PUT of the same path is **overwrite-by-default = idempotent last-write-wins** (resume for free) |
| Same file uploaded twice (double-click) | ✅ unchanged (C1) | unique `file_registry.sha256_checksum` at finalize — second insert fails with pointer to the original |
| Duplicate delivery to intake | ✅ unchanged (C4) | partial-unique `file_transfers (file_id, direction) WHERE COMPLETED` — DB-enforced |
| Move raced by two workers | ✅ unchanged | atomic claim on PENDING transfer; loser matches 0 docs |
| Upload succeeded but file never landed | ✅ unchanged (C6) | **ack-after-durable-PUT**: success only after the Files API PUT returns 201/200 **and** `file_registry` exists and `FILE_ARRIVED` is persisted — one Mongo transaction; HEAD re-verifies size at complete |
| Worker crash mid-move | ✅ | staging copy is deleted **only after** the destination PUT + verify succeed — worst case = a duplicated copy retried idempotently, never a lost file; re-drive from `file_transfers` |
| Throughput ceiling | ⚠️ bounded | Files API is an HTTPS control-plane endpoint with rate limits — fine for thousands of small files/hour across pods (keep-alive, streaming, no buffering); **benchmark at R2** and if sustained ingest grows 10×, move steps 4–6 into a Databricks-side job (same collections, same contracts) — no model change |
| Directory listing stress | ✅ avoided | everything is path-addressable; no `GET /fs/directories` scans in the hot path; janitor cleans per-session dirs |

**What the janitor becomes:** orphan-chunk cleanup → **abandoned-session directory cleanup** (`DELETE /api/2.0/fs/directories/sessions/<ses_id>` for sessions TTL-expired with no finalize). Simpler than the GridFS equivalent.

**Purge (Phase 3) becomes simpler:** 1 `DELETE` per file on the intake/quarantine volume copy (+ archive rules) instead of a chunk-doc transaction; `file_registry.gridfs_purged_at` is renamed **`volume_purged_at`**. The gate is untouched: retention exceeded **AND** fresh MATCHED **AND** no legal hold.

## 5. Model Deltas (applied in LLD v1.3 + Steps 1–2)

| Change | Where | Notes |
| --- | --- | --- |
| `file_registry.staged_path` | **primary bytes pointer** | `staging/sessions/<ses_id>/<original_name>`; required for portal files (app-enforced) |
| `file_registry.gridfs_file_id` | objectId → **nullable, legacy** | null for all Phase-1 files; kept for any future source that opts into GridFS (none planned) |
| `file_registry.gridfs_purged_at` → **`volume_purged_at`** | rename | purge tombstone now tracks the volume copy; partial purge-candidate index updated accordingly |
| `upload_sessions.parts[]` | `{file_id, staged_path, status, bytes_received}` | replaces `{gridfs_file_id, chunks_received}` |
| `volume_map.destination_type` enum += `STAGING` | + seed row `*` → STAGING root | new catch-all staging mapping |
| `file_lifecycle.location` enum += `STAGING_VOLUME` | first lifecycle entry per file | GRIDFS value retained for legacy compatibility |
| GridFS section (LLD §8) | **deprecated** | retained for reference; no `fs` bucket is created in Phase 1 |
| ER diagram `fs.files` link | legacy annotation | `file_registry }|--|| fs.files` becomes optional/legacy |

**Unchanged collections:** `event_log`, `validation_rules`, `schema_versions`, `routing_policies`, `notifications`, `audit_log`, `reconciliation_*`, `databricks_ingestion_results` — the metadata spine is storage-agnostic. `file_transfers.from_location`/`to_location` already hold paths (`volume:<path>` for Phase 1 instead of `gridfs:<objectId>`).

## 6. Impact on Existing Docs

* **HLD:** D2 (single GridFS bucket) is reversed by this addendum — backup/DR story simplifies to *Mongo metadata (PITR) + volume snapshots/lifecycle*; growth table's `fs.chunks` share disappears. Storage topology now: **staging / intake / quarantine volumes** per environment.
* **LLD:** bumped to v1.3 with the deltas above + change-log entry.
* **Build Step 1:** `volume_map` validator + seeds gain `STAGING`; expected seed count 1/5/5/8/3/**7**.
* **Build Step 2:** `file_registry` validator (gridfs nullable, renamed tombstone), `upload_sessions.parts`, `file_lifecycle` enum; **no `fs.files` secondary index** — dropped.
* **Purge Job Design:** banner added — 'GridFS copy' reads as 'volume copy'; field renamed; gate logic unchanged.
* **Resilience Addendum:** C6/C7 mechanics map 1:1 (ack-after-durable-write now = ack-after-durable-PUT; orphan chunks → abandoned session dirs). No scenario loses coverage.

## 7. Open Items for Review

1. **Volume layout** — ✅ **RESOLVED (2026-11-09):** schema `mdd_platform.fvr` + managed volumes `staging`, `intake`, `quarantine` created and smoke-tested (session-dir write → intake move → cleanup all pass). Roots: `/Volumes/mdd_platform/fvr/staging` (subfolder `sessions/`), `/Volumes/mdd_platform/fvr/intake` (subfolder per file type), `/Volumes/mdd_platform/fvr/quarantine` (subfolder `unidentified`).
2. **Rate-limit budget** — expected peak uploads/hour; drives whether R2 needs a load test or just a smoke test.
3. **Big-file threshold** — agree ~100 MB as the point where chunked/session upload needs designing (none today).
4. **Databricks-side execution option** — at 10× growth, steps 4–6 (validate/move) move into a Lakeflow job; decide at R2 with real numbers, not now.
5. **Environment volumes** — dev/staging/prod volume roots + service principal per environment.